In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;


# Module 3: Feature Store

이 모듈에서는 **현실적인 ML 피처 파이프라인**을 구축합니다.

### 핵심 구조

```
TPC-H 원본 (ORDERS + LINEITEM + CUSTOMER)
    │
    ▼  [일별 피처 스냅샷 생성]
CUSTOMER_FEATURES_DAILY (10,000 고객 × 180일 = 180만 행)
    │
    ▼  [Feature Store 등록: Managed FV + timestamp_col]
customer_ltv_features (Dynamic Table, 자동 갱신)
    │
    ├── 학습: generate_dataset() + Spine(EVENT_TIMESTAMP) → 불변 Dataset
    │
    └── 추론: Managed FV → LIVE_PREDICTED_LTVS (자동 재추론)
```

### Snowflake 객체 매핑

| Feature Store 개념 | 실제 Snowflake 객체 | 설명 |
|---|---|---|
| Feature Store | Schema | 피처를 담는 스키마 |
| Entity | Tag | 피처의 기준 키 (C_CUSTKEY) |
| Managed Feature View | Dynamic Table | 자동 갱신, timestamp_col로 Point-in-Time 지원 |
| Dataset | Dataset 객체 | 불변 학습 데이터 스냅샷 |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
import snowflake.snowpark.types as T

from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml import dataset

session = get_active_session()
print('Session ready')


## 1. 일별 피처 스냅샷 테이블 생성

현실에서는 매일 피처가 갱신됩니다. TPC-H 데이터로 이를 시뮬레이션합니다:
- 10,000 고객 샘플 × 180일 (1997-01-01 ~ 1997-06-30)
- 각 날짜 기준으로 **그 날까지의 누적** 피처를 계산
- 타겟(`FUTURE_12M_REVENUE`)은 별도 계산 (1997.07~1998.06 구매금액)


In [ ]:
# ── 일별 피처 스냅샷 생성 (SQL) ──────────────────────────────────────────────
# 10,000 고객 × 180일 = 약 180만 행
# 각 FEATURE_DATE 기준으로 그 날까지의 누적 주문 통계를 계산합니다.

create_snapshot_sql = """
CREATE OR REPLACE TABLE DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY AS
WITH
-- 10,000 고객 샘플링 (RANDOM 시드 고정)
sampled_customers AS (
    SELECT C_CUSTKEY, C_MKTSEGMENT, C_ACCTBAL, C_NATIONKEY
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
    ORDER BY C_CUSTKEY
    LIMIT 10000
),
-- 180일 날짜 시퀀스 (1997-01-01 ~ 1997-06-30)
date_spine AS (
    SELECT DATEADD('day', SEQ4(), '1997-01-01'::DATE) AS FEATURE_DATE
    FROM TABLE(GENERATOR(ROWCOUNT => 181))
    WHERE DATEADD('day', SEQ4(), '1997-01-01'::DATE) <= '1997-06-30'
),
-- LINEITEM 집계 (주문 단위)
lineitem_agg AS (
    SELECT L_ORDERKEY,
           SUM(L_EXTENDEDPRICE * (1 - L_DISCOUNT)) AS LINE_REVENUE,
           AVG(L_DISCOUNT) AS LINE_AVG_DISCOUNT,
           AVG(L_QUANTITY) AS LINE_AVG_QUANTITY
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM
    GROUP BY L_ORDERKEY
),
-- 각 (고객, 날짜) 조합에 대해 그 날까지의 누적 피처 계산
daily_features AS (
    SELECT
        c.C_CUSTKEY,
        d.FEATURE_DATE,
        c.C_MKTSEGMENT,
        c.C_ACCTBAL,
        c.C_NATIONKEY,
        COUNT(o.O_ORDERKEY) AS TOTAL_ORDERS,
        COALESCE(AVG(o.O_TOTALPRICE), 0) AS AVG_ORDER_VALUE,
        COALESCE(SUM(la.LINE_REVENUE), 0) AS TOTAL_REVENUE,
        COALESCE(AVG(la.LINE_AVG_DISCOUNT), 0) AS AVG_DISCOUNT,
        COALESCE(AVG(la.LINE_AVG_QUANTITY), 0) AS AVG_QUANTITY,
        COALESCE(DATEDIFF('day', MAX(o.O_ORDERDATE), d.FEATURE_DATE), 9999) AS DAYS_SINCE_LAST_ORDER
    FROM sampled_customers c
    CROSS JOIN date_spine d
    LEFT JOIN SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS o
        ON o.O_CUSTKEY = c.C_CUSTKEY AND o.O_ORDERDATE <= d.FEATURE_DATE
    LEFT JOIN lineitem_agg la
        ON la.L_ORDERKEY = o.O_ORDERKEY
    GROUP BY c.C_CUSTKEY, d.FEATURE_DATE, c.C_MKTSEGMENT, c.C_ACCTBAL, c.C_NATIONKEY
)
SELECT * FROM daily_features
ORDER BY C_CUSTKEY, FEATURE_DATE
"""

print('Creating CUSTOMER_FEATURES_DAILY (10K customers x 180 days)...')
print('This may take a few minutes.')
session.sql(create_snapshot_sql).collect()

row_count = session.table('DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY').count()
print(f'Done! {row_count:,} rows created')
session.table('DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY').show(5)


In [ ]:
# ── 타겟 컬럼 추가 (FUTURE_12M_REVENUE) ──────────────────────────────────────
# 예측 윈도우: 1997-07-01 ~ 1998-06-30
# 각 고객의 이 기간 구매금액 합계 (미구매 시 0)

add_target_sql = """
CREATE OR REPLACE TABLE DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY AS
WITH target AS (
    SELECT O_CUSTKEY,
           SUM(O_TOTALPRICE) AS FUTURE_12M_REVENUE
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
    WHERE O_ORDERDATE BETWEEN '1997-07-01' AND '1998-06-30'
    GROUP BY O_CUSTKEY
)
SELECT
    f.*,
    COALESCE(t.FUTURE_12M_REVENUE, 0) AS FUTURE_12M_REVENUE
FROM DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY f
LEFT JOIN target t ON f.C_CUSTKEY = t.O_CUSTKEY
"""

print('Adding FUTURE_12M_REVENUE target column...')
session.sql(add_target_sql).collect()
print('Done!')

# 확인
df_daily = session.table('DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY')
print(f'\nTotal rows: {df_daily.count():,}')
print(f'Columns: {df_daily.columns}')
df_daily.filter(F.col('FEATURE_DATE') == '1997-06-30').show(5)


## 2. Feature Store 초기화 & Entity 등록


In [ ]:
# Feature Store 초기화
fs = FeatureStore(
    session=session,
    database='DEMO',
    name='ML_DEMO',
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

# Entity 등록
customer_entity = Entity(name='CUSTOMER', join_keys=['C_CUSTKEY'])
try:
    fs.register_entity(customer_entity)
except Exception:
    pass

print('Feature Store ready')
print(f'Entities: {fs.list_entities().to_pandas()["NAME"].tolist()}')


## 3. Managed Feature View 등록 (Point-in-Time 지원)

**핵심**: `timestamp_col="FEATURE_DATE"`를 지정하면 Feature Store가 ASOF JOIN으로
Spine의 `EVENT_TIMESTAMP` 이전 가장 최근 피처를 자동 조회합니다.

```
Spine: C_CUSTKEY=1001, EVENT_TIMESTAMP='1997-06-30'
    → CUSTOMER_FEATURES_DAILY에서 FEATURE_DATE <= '1997-06-30' 인 가장 최근 행 반환
    → Data Leakage 자동 방지!
```


In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*incrementally refreshed.*')

# Feature DataFrame 준비 (타겟 제외 — 피처만)
feature_df = session.table('DEMO.ML_DEMO.CUSTOMER_FEATURES_DAILY').select(
    'C_CUSTKEY', 'FEATURE_DATE',
    'C_MKTSEGMENT', 'C_ACCTBAL', 'C_NATIONKEY',
    'TOTAL_ORDERS', 'AVG_ORDER_VALUE', 'TOTAL_REVENUE',
    'AVG_DISCOUNT', 'AVG_QUANTITY', 'DAYS_SINCE_LAST_ORDER'
)

# Managed Feature View 정의
customer_ltv_fv = FeatureView(
    name='customer_ltv_features',
    entities=[customer_entity],
    feature_df=feature_df,
    timestamp_col='FEATURE_DATE',  # Point-in-Time 지원!
    refresh_freq='1 hour',
    desc='Customer LTV features (daily snapshots, Point-in-Time enabled)'
)

# 등록
registered_fv = fs.register_feature_view(
    feature_view=customer_ltv_fv,
    version='1',
    overwrite=True
)

print('Managed Feature View registered!')
print(f'  Name: {registered_fv.name}')
print(f'  Version: {registered_fv.version}')
print(f'  Timestamp col: FEATURE_DATE')
print(f'  Refresh freq: {registered_fv.refresh_freq}')


In [ ]:
# 등록된 Feature View 목록 확인
print('Registered Feature Views:')
fvs = fs.list_feature_views().to_pandas()
print(fvs[['NAME','VERSION','REFRESH_FREQ']].to_string(index=False) if 'REFRESH_FREQ' in fvs.columns else fvs.to_string(index=False))


## 4. 학습 Dataset 생성 (Point-in-Time JOIN)

**Spine DataFrame**을 만들어서 `generate_dataset()`을 호출합니다:
- `C_CUSTKEY`: 어떤 고객
- `EVENT_TIMESTAMP`: 어떤 시점 기준으로 피처를 가져올지
- `FUTURE_12M_REVENUE`: 타겟 (정답)

Feature Store가 `FEATURE_DATE <= EVENT_TIMESTAMP`인 가장 최근 행을 자동 조회합니다.


In [ ]:
# Spine 생성: 10,000 고객 × EVENT_TIMESTAMP='1997-06-30'
# 타겟(FUTURE_12M_REVENUE)도 Spine에 포함

spine_sql = """
SELECT
    c.C_CUSTKEY,
    '1997-06-30'::TIMESTAMP_NTZ AS EVENT_TIMESTAMP,
    COALESCE(t.FUTURE_12M_REVENUE, 0) AS FUTURE_12M_REVENUE
FROM (
    SELECT C_CUSTKEY FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER
    ORDER BY C_CUSTKEY LIMIT 10000
) c
LEFT JOIN (
    SELECT O_CUSTKEY, SUM(O_TOTALPRICE) AS FUTURE_12M_REVENUE
    FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
    WHERE O_ORDERDATE BETWEEN '1997-07-01' AND '1998-06-30'
    GROUP BY O_CUSTKEY
) t ON c.C_CUSTKEY = t.O_CUSTKEY
"""

spine_df = session.sql(spine_sql)
print(f'Spine: {spine_df.count():,} rows')
print('Columns:', spine_df.columns)
spine_df.show(5)


In [ ]:
# generate_dataset()으로 Point-in-Time 학습 데이터 생성
# Feature Store가 ASOF JOIN으로 EVENT_TIMESTAMP 이전 최신 피처를 자동 매칭

training_ds = fs.generate_dataset(
    name='customer_ltv_training',
    version='v1',
    spine_df=spine_df,
    features=[registered_fv],
    spine_timestamp_col='EVENT_TIMESTAMP',
    spine_label_cols=['FUTURE_12M_REVENUE'],
    desc='LTV training dataset (Point-in-Time: 1997-06-30)'
)

print('Dataset generated!')
print(f'  Name: customer_ltv_training')
print(f'  Version: v1')

# 확인
train_df = training_ds.read.to_snowpark_dataframe()
print(f'  Rows: {train_df.count():,}')
print(f'  Columns: {train_df.columns}')
train_df.show(5)


> **Point-in-Time이 보장하는 것:**
> - Spine의 `EVENT_TIMESTAMP='1997-06-30'`
> - Feature Store가 `FEATURE_DATE <= 1997-06-30`인 가장 최근 스냅샷을 반환
> - 1997-07-01 이후 데이터는 절대 피처에 포함되지 않음 → **Data Leakage 완전 방지**
>
> 이것이 Feature Store의 핵심 가치입니다.


## 5. 정리

| 단계 | 생성된 객체 | 용도 |
|------|------------|------|
| 일별 스냅샷 | `CUSTOMER_FEATURES_DAILY` | Feature View 소스 |
| Managed FV | `customer_ltv_features@1` | 추론 + Point-in-Time 학습 |
| Dataset | `customer_ltv_training@v1` | 불변 학습 데이터 (재현 가능) |

### 다음 단계

Module 4에서 `customer_ltv_training@v1` Dataset으로 모델을 학습합니다.
